In [1]:
from typing import List, TypedDict,Literal
from pydantic import BaseModel, Field
import  time
from pypdf import PdfReader
#from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai  import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings 
from langchain_groq import ChatGroq

from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7244\1082459604.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [ ]:
#---------------LOAD DOCS-----------------------------------
#Helper function to replicate PyPDFLoader
def load_pdf_as_documents(file_path: str) -> list[Document]:
    try:
        reader = PdfReader(file_path)
        documents = []

        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            # Replicates LangChain's default metadata schema for loaders
            metadata = {"source": file_path, "page": i}
            documents.append(Document(page_content=text, metadata=metadata))

        return documents
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


# Combining docs identically
docs = (
    load_pdf_as_documents("./documents/Company_Policies.pdf")
    + load_pdf_as_documents("documents/Company_Profile.pdf")
    + load_pdf_as_documents("documents/Product_and_Pricing.pdf")
)

#--------------------------CHUNKING--------------------------------
#chunking the documents using Recursive Character text splitter
chunks = RecursiveCharacterTextSplitter(chunk_size =500, chunk_overlap =100).split_documents(docs)

#------------------------------EMBED DOCS-----------------------------------
#Generating Embeddings for given docs using text-embedding-004 from google generative ai embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-001",# or 'models/gemini-embedding-exp-03-07'
    task_type="retrieval_document"# Options: retrieval_query, retrieval_document, semantic_similarity, etc.
)

#----------------------------------VECTOR STORE-----------------------------------
#Storing in vector database using  FAISS db
vector_store = FAISS.from_documents(chunks,embeddings)

#-------------------------RETRIEVER-----------------------------
#Creating a retriever using vector store
retriever =  vector_store.as_retriever(search_kwargs ={"k":4})

In [ ]:
#---------------------------LLM-------------------------------
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
#llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash",temperature = 0)

In [ ]:
#-------------------- Graph State ---------------------------------------
class State(TypedDict):
    #user query 
    question : str

    #to check if it needs retrieval from docs
    need_retrieval : bool
    
    docs : List[Document]
    relevant_docs: List[Document]
    
    context : str
    answer : str

    # web query simple
    web_query: str

In [ ]:
#------------------------------NODES DEFINITION---------------------------------------

#----------------------------1. DECIDE RETRIEVAL --------------------------------------------

#using pydantic 
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)

# Note: no `.content` for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state:State):
    decision : RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages( question = state['question'])
    )
    return {"need_retrieval": decision.should_retrieve}


def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"

In [ ]:
#--------------------------2. DIRECT ANSWER - No need of Retrieval -------------------------------

direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human","{question}")
    ]
)

def generate_direct(state:State):
    output = llm.invoke(
        direct_generation_prompt.format_messages(question = state['question'])
    )
    return {"answer" :output.content}
    

In [ ]:
#----------------------------3. RETRIEVE -----------------------------

def retrieve(state:State):
    query = state.get("retrieval_query") or state["question"]
    return {"docs": retriever.invoke(query)}

In [ ]:
#---------------------------------4. RELEVANCE FILTER (strict)--------------------

#to get the relevance Decision Pydantic model
class RelevanceDecision(BaseModel):
    is_relevant: bool = Field(
        ...,
        description="True if the document helps answer the question, else False."
    )

is_relevant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are judging document relevance.\n"
            "Return JSON that matches this schema:\n"
            "{{'is_relevant': boolean}}\n\n"
            "A document is relevant if it contains information useful for answering the question."
        ),
        (
            "human",
            "Question:\n{question}\n\nDocument:\n{document}"
        ),
    ]
)

relevance_llm = llm.with_structured_output(RelevanceDecision)

def is_relevant(state:State):
    relevant_docs : List[Document] = []

    for doc in state["docs"]:
        decision : RelevanceDecision = relevance_llm.invoke(
            is_relevant_prompt.format_messages(
                question = state["question"],
                document = doc.page_content
            )
        )
        if decision.is_relevant:
            relevant_docs.append(doc)

        return {"relevant_docs":relevant_docs}

def route_after_relevance(state:State)-> Literal["generate_from_context", "no_answer_found"]:
    if state.get("relevant_docs") and len(state["relevant_docs"]) > 0:
        return "generate_from_context"
    return "rewrite_query"

In [ ]:
#----------------------5. GENERATING FROM CONTEXT ----------------------

rag_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a business RAG assistant.\n"
            "Answer the user's question using ONLY the provided context.\n"
            "If the context does not contain enough information, say:\n"
            "'No relevant document found.'\n"
            "Do not use outside knowledge.\n"
        ),
        (
            "human",
            "Question:\n{question}\n\n"
            "Context:\n{context}\n"
        ),
    ]
)

def generate_from_context(state:State):
    # Stuff relevant docs into one block
    context = "\n\n ===\n\n".join([d.page_content for d in state.get("relevant_docs", [])]).strip()
    if not context:
        return {"answer": "No relevant document found.", "context": ""}
    output = llm.invoke(rag_generation_prompt.format_messages(question= state['question'],context = context))
    return {"answer":output.content,"context":context}




In [ ]:
#---------------------------------6. WEB QUERY-------------------------------
class WebQuery(BaseModel):
    query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules:\n"
            "- Keep it short (6–14 words).\n"
            "- If the question implies recency, add (last 30 days).\n"
            "- Do NOT answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human", "Question: {question}"),
    ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)

def rewrite_query_node(state: State):
    out = rewrite_chain.invoke({"question": state["question"]})
    return {"web_query": out.query}

tavily = TavilySearchResults(max_results=5)

def web_search_node(state: State):
    q = state.get("web_query") or state["question"]
    results = tavily.invoke({"query": q})

    docs = []
    for r in results or []:
        title = r.get("title", "")
        url = r.get("url", "")
        content = r.get("content", "") or r.get("snippet", "")
        text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
        docs.append(
            Document(
                page_content=text,
                metadata={"source": "web", "url": url, "title": title},
            )
        )

    return {"docs": docs}

In [ ]:
def no_relevant_docs(state: State):
    return {"answer": "No relevant document found.", "context": ""}

In [ ]:
#---------------------------------BUILDING GRAPH-------------------------------------
graph = StateGraph(State)

#-------------------------------------Nodes---------------------------------------------------

#1. decide and generate if no retrieval needed 
graph.add_node("decide_retrieval", decide_retrieval)
graph.add_node("generate_direct", generate_direct)
graph.add_node("retrieve", retrieve)

#2. if retrieval needed then generate context and  check for ans available in docs
graph.add_node("is_relevant", is_relevant)
graph.add_node("generate_from_context", generate_from_context)

#3. adding web search node 
graph.add_node("rewrite_query", rewrite_query_node)
graph.add_node("web_search", web_search_node)


#----------------------------------------Edges--------------------------------------------------

graph.add_edge(START, "decide_retrieval")
graph.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {"generate_direct": "generate_direct", "retrieve": "retrieve"},
)
graph.add_edge("generate_direct", END)

#Retrieve -> relevance 

graph.add_edge("retrieve", "is_relevant")

# relevance router: if relevant → generate, else → rewrite_query
graph.add_conditional_edges(
    "is_relevant",
    route_after_relevance,
    {
        "generate_from_context": "generate_from_context",
        "rewrite_query": "rewrite_query"
    },
)

# web fallback path

graph.add_edge("rewrite_query", "web_search")
graph.add_edge("web_search", "is_relevant")  #loop back

graph.add_edge("generate_from_context", END)

#--------------------------------COMPILE GRAPH ----------------------------
app = graph.compile()

#---------------------------GRAPH STRUCTURE ----------------------------------
app

In [ ]:
#----------------------------10. RUN GRAPH ----------------------------------

result = app.invoke(
    {
        "question": "Who won the Aus vs Zim World T20 match 2026 and who was the top scorer",
        "docs": [],
        "relevant_docs": [],
        "context": "",
        "answer": "",
    }
)

print(result["answer"])

In [ ]:
for doc in result['relevant_docs']:
    print(doc.page_content)
    print("*"*100)